In [1]:
import numpy as np
import keras
import matplotlib.pyplot as plt

I0000 00:00:1776237875.636101    4018 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
from hgq.utils.sugar import Dataset

input_shape = (28, 28, 1)

# Load the data and split it between train and test sets
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

# Binarize the images (convert to black and white) instead of scaling
X_train, X_test = (X_train > 127).astype(np.float32), (X_test > 127).astype(np.float32)

# Scale images to the [0, 1] range
#X_train = X_train.astype("float32") / 255
#X_test = X_test.astype("float32") / 255

#(Not required for HGQ)
# Make sure images have shape (28, 28, 1)
#X_train = np.expand_dims(X_train, -1)
#X_test = np.expand_dims(X_test, -1)
#print("X_train shape:", X_train.shape)
#print(X_train.shape[0], "train samples")
#print(X_test.shape[0], "test samples")

#(using SparseCategoricalCrossentropy instead)
# Convert class vectors to binary class matrices
#y_train = keras.utils.to_categorical(y_train, 10)
#y_test = keras.utils.to_categorical(y_test, 10)

#Shuffles the data
rng = np.random.default_rng(42)
order = rng.permutation(len(X_train))
X_train, y_train = X_train[order], y_train[order]

#Splits the training data into a training and validation set, equivalent to validation_split=0.1 in model.fit

N_train = int(0.9 * len(X_train))
X_train, X_val = X_train[:N_train], X_train[N_train:]
y_train, y_val = y_train[:N_train], y_train[N_train:]

# shuffle=True responsible for per Epoch variation
dataset_train = Dataset(X_train, y_train, batch_size=5000, device='gpu:0', shuffle=True)
dataset_val = Dataset(X_val, y_val, batch_size=5000, device='gpu:0')

W0000 00:00:1776237878.905674    4018 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [3]:
from keras.models import Sequential
from keras.optimizers import Adam
from keras.losses import SparseCategoricalCrossentropy
from hgq.config import QuantizerConfig, QuantizerConfigScope
from hgq.layers import QDense, QConv2D, QMaxPooling2D, QAveragePooling2D, QSoftmax, QEinsumDenseBatchnorm
from hgq.utils.sugar import FreeEBOPs, PBar

In [4]:
# overflow modes:
#'sat_sym' clips to max value if overflow, 'wrap' wraps around (modulo) if overflow
scope_weights_and_biases = QuantizerConfigScope(
    place='all', 
    default_q_type='kbi', 
    heterogeneous_axis=None, 
    homogeneous_axis=(),
    overflow_mode='sat_sym'
)

scope_activations = QuantizerConfigScope(
    place='datalane', 
    default_q_type='kif', 
    heterogeneous_axis=None, 
    homogeneous_axis=(0, 1),
    overflow_mode='wrap'
)

In [ ]:
def build_model(beta0=1e-6):
    # 1. Define the input quantizer explicitly 
    iqc = QuantizerConfig(trainable=False, i0=1, k0=0, f0=0)
    
    with scope_weights_and_biases, scope_activations:
        model = Sequential([
            keras.layers.Input(shape=(28, 28, 1)), 
            
            # 2. Use the Input Quantizer config here if QConv2D supports it, 
            # or use an AveragePool/Identity layer first to set the scale.
            QConv2D(32, beta0=beta0, kernel_size=(3, 3), name='conv_0', iq_conf=iqc),
            
            QMaxPooling2D(pool_size=(2, 2), name='maxpool_0'),
            
            QConv2D(32, beta0=beta0, kernel_size=(3, 3), name='conv_1'),
            keras.layers.BatchNormalization(), # CRITICAL
            keras.layers.Activation('relu'),
            
            QMaxPooling2D(pool_size=(2, 2), name='maxpool_1'),
            
            keras.layers.Flatten(), 
            keras.layers.Dropout(0.4),
            QDense(10, beta0=beta0, name='output'), # No softmax here
        ])
    return model
model = build_model(beta0=1e-7)
loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
opt = keras.optimizers.Adam(learning_rate=0.001)

model.summary()

model.compile(optimizer=opt, loss=loss, metrics=['accuracy'], jit_compile=True, steps_per_execution=8)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv_0 (QConv2D)                │ (None, 26, 26, 32)     │         3,635 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 26, 26, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 26, 26, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool_0 (QMaxPooling2D)       │ (None, 13, 13, 32)     │         2,499 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_1 (QConv2D)                │ (None, 11, 11, 32)     │        38,243 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 11, 11, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 11, 11, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool_1 (QMaxPooling2D)       │ (None, 5, 5, 32)       │         1,059 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 800)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 800)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (QDense)                 │ (None, 10)             │        32,046 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 77,738 (245.18 KB)

 Trainable params: 54,463 (212.75 KB)

 Non-trainable params: 23,275 (32.43 KB)

In [5]:
def build_model(beta0=1e-6):
    with scope_weights_and_biases, scope_activations:

        iqc = QuantizerConfig(trainable=False, i0=1, k0=0, f0=0)
        inp = keras.Input(shape=input_shape)
        out = QAveragePooling2D(pool_size=2, padding='valid', iq_conf=iqc)(inp)
        out = keras.layers.Flatten()(out)
        out = QEinsumDenseBatchnorm(
            'bc,cC->bC', 24, name='t1', bias_axes='C', activation='relu', kernel_regularizer=keras.regularizers.L2(1e-3)
        )(out)
        out = QEinsumDenseBatchnorm(
            'bc,cC->bC', 24, name='t2', bias_axes='C', activation='relu', kernel_regularizer=keras.regularizers.L2(1e-3)
        )(out)
        out = QEinsumDenseBatchnorm('bc,cC->bC', 24, name='t3', bias_axes='C', activation='relu')(out)
        out = QDense(10, name='out')(out)
    return keras.Model(inputs=inp, outputs=out)

model = build_model(beta0=1e-6)
loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
opt = keras.optimizers.Adam(learning_rate=0.001)

model.summary()

model.compile(optimizer=opt, loss=loss, metrics=['accuracy'], jit_compile=True, steps_per_execution=8)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_average_pooling2d             │ (None, 14, 14, 1)      │            87 │
│ (QAveragePooling2D)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 196)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ t1 (QEinsumDenseBatchnorm)      │ (None, 24)             │        18,990 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ t2 (QEinsumDenseBatchnorm)      │ (None, 24)             │         2,478 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ t3 (QEinsumDenseBatchnorm)      │ (None, 24)             │         2,478 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ out (QDense)                    │ (None, 10)             │         1,006 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,039 (79.62 KB)

 Trainable params: 18,610 (72.70 KB)

 Non-trainable params: 6,429 (6.92 KB)

In [6]:
pbar = PBar('loss: {loss:.3f}/{val_loss:.3f} - acc: {accuracy:.3f}/{val_accuracy:.3f}')


ebops = FreeEBOPs()
nan_terminate = keras.callbacks.TerminateOnNaN()
callbacks = [ebops, pbar, nan_terminate]

In [7]:
model.fit(dataset_train, epochs=200, validation_data=dataset_val, callbacks=callbacks, verbose=0)

  0%|          | 0/200 [00:00<?, ?epoch/s]I0000 00:00:1776237899.339841    4018 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1776237903.931935    4120 service.cc:153] XLA service 0x7f3950026ed0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776237903.931982    4120 service.cc:161]   StreamExecutor [0]: Host, Default Version (Driver: 0.0.0; Runtime: 0.0.0; Toolkit: 0.0.0; DNN: 0.0.0)
I0000 00:00:1776237904.102072    4120 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1776237905.898855    4120 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
loss: 0.789/0.254 - acc: 0.943/0.927 - EBOPs: 58,399:  63%|██████▎   | 126/200 [00:37<00:18,  4.08epoch/s]

KeyboardInterrupt: 

In [10]:
score = model.evaluate(X_test, y_test, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])

Test loss: 0.3598361015319824
Test accuracy: 0.8913999795913696


In [ ]:
model.save('/home/slopin/DAT255-project/Modeller/mnist_baseline.keras')